In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd

# Define your folders
folder1 = '/content/drive/MyDrive/Data/VED-dataset/VED_DynamicData_Part1'
folder2 = '/content/drive/MyDrive/Data/VED-dataset/VED_DynamicData_Part2'

# Function to get row counts for each CSV file
def get_row_counts(folder_path):
    file_counts = {}
    for file in sorted(os.listdir(folder_path)):  # sorted for consistent order
        if file.endswith('.csv'):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)
            file_counts[file] = len(df)
    return file_counts

# Get row counts for both folders
counts_folder1 = get_row_counts(folder1)
counts_folder2 = get_row_counts(folder2)

# Print Folder 1 details
print("Folder 1:")
total_folder1 = 0
for file, count in counts_folder1.items():
    print(f"{file}: {count:,} rows")  # Add commas in number
    total_folder1 += count
print(f"Total rows in Folder 1: {total_folder1:,}\n")

# Print Folder 2 details
print("Folder 2:")
total_folder2 = 0
for file, count in counts_folder2.items():
    print(f"{file}: {count:,} rows")
    total_folder2 += count
print(f"Total rows in Folder 2: {total_folder2:,}\n")

# Print combined total
grand_total = total_folder1 + total_folder2
print(f"Grand Total (Folder 1 + Folder 2): {grand_total:,}")

Folder 1:
VED_171101_week.csv: 489,414 rows
VED_171108_week.csv: 535,198 rows
VED_171115_week.csv: 602,092 rows
VED_171122_week.csv: 474,329 rows
VED_171129_week.csv: 531,856 rows
VED_171206_week.csv: 589,154 rows
VED_171213_week.csv: 670,084 rows
VED_171220_week.csv: 539,468 rows
VED_171227_week.csv: 311,927 rows
VED_180103_week.csv: 393,498 rows
VED_180110_week.csv: 550,780 rows
VED_180117_week.csv: 526,771 rows
VED_180124_week.csv: 456,757 rows
VED_180131_week.csv: 479,522 rows
VED_180207_week.csv: 419,846 rows
VED_180214_week.csv: 389,642 rows
VED_180221_week.csv: 285,924 rows
VED_180228_week.csv: 402,041 rows
VED_180307_week.csv: 398,405 rows
VED_180314_week.csv: 420,510 rows
VED_180321_week.csv: 402,133 rows
VED_180328_week.csv: 367,606 rows
Total rows in Folder 1: 10,236,957

Folder 2:
VED_180404_week.csv: 461,047 rows
VED_180411_week.csv: 434,636 rows
VED_180418_week.csv: 486,657 rows
VED_180425_week.csv: 451,633 rows
VED_180502_week.csv: 426,242 rows
VED_180509_week.csv: 421,6

In [ ]:
first_file = sorted(os.listdir(folder1))[0]  # Get first CSV file name
first_file_path = os.path.join(folder1, first_file)
print(f"First 5 rows of {first_file}:\n")
df_first = pd.read_csv(first_file_path)
print(df_first.head())

First 5 rows of VED_171101_week.csv:

     DayNum  VehId  Trip  Timestamp(ms)  Latitude[deg]  Longitude[deg]  \
0  1.586651      8   706              0      42.277558       -83.69875   
1  1.586651      8   706            200      42.277558       -83.69875   
2  1.586651      8   706           1100      42.277558       -83.69875   
3  1.586651      8   706           2100      42.277558       -83.69875   
4  1.586651      8   706           4200      42.277558       -83.69875   

   Vehicle Speed[km/h]  MAF[g/sec]  Engine RPM[RPM]  Absolute Load[%]  ...  \
0                 40.0   22.129999           2285.0         49.019608  ...   
1                 40.0   22.129999           2285.0         67.450981  ...   
2                 45.0   22.129999           2285.0         67.450981  ...   
3                 47.0    6.150000           2744.0         67.450981  ...   
4                 48.0   21.440001           1982.0         67.450981  ...   

   Air Conditioning Power[kW]  Air Conditioning 

In [ ]:
# Read static data
df_ice_hev = pd.read_excel('/content/drive/MyDrive/Data/VED-dataset/VED_Static_Data_ICE&HEV.xlsx')
df_phev_ev = pd.read_excel('/content/drive/MyDrive/Data/VED-dataset/VED_Static_Data_PHEV&EV.xlsx')

# Strip column names of spaces
df_ice_hev.columns = df_ice_hev.columns.str.strip()
df_phev_ev.columns = df_phev_ev.columns.str.strip()

# Check actual column names
print("ICE&HEV columns:", df_ice_hev.columns.tolist())
print("PHEV&EV columns:", df_phev_ev.columns.tolist())

ICE&HEV columns: ['VehId', 'Vehicle Type', 'Vehicle Class', 'Engine Configuration & Displacement', 'Transmission', 'Drive Wheels', 'Generalized_Weight']
PHEV&EV columns: ['VehId', 'EngineType', 'Vehicle Class', 'Engine Configuration & Displacement', 'Transmission', 'Drive Wheels', 'Generalized_Weight']


Separating the gasoline vehicles and HEV & PHEV vehicles data

In [ ]:
# Extract VehIds
gasoline_ids = df_ice_hev[df_ice_hev['Vehicle Type'] == 'ICE']['VehId'].tolist()
hev_gas_ids = list(df_ice_hev[df_ice_hev['Vehicle Type'] == 'HEV']['VehId']) + \
              list(df_phev_ev[df_phev_ev['EngineType'] == 'PHEV']['VehId'])

# Create new folders
gas_folder = '/content/drive/MyDrive/Data/VED-dataset/VED_Gasoline'
hev_folder = '/content/drive/MyDrive/Data/VED-dataset/VED_HEV_PHEV'

os.makedirs(gas_folder, exist_ok=True)
os.makedirs(hev_folder, exist_ok=True)

# Function to filter CSVs based on VehId and save
def filter_and_save(folder_path, output_gas_folder, output_hev_folder):
    for file in sorted(os.listdir(folder_path)):
        if file.endswith('.csv'):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)

            # Filter gasoline vehicles
            df_gas = df[df['VehId'].isin(gasoline_ids)]
            if not df_gas.empty:
                df_gas.to_csv(os.path.join(output_gas_folder, file), index=False)

            # Filter HEV/PHEV using gasoline
            df_hev = df[df['VehId'].isin(hev_gas_ids)]
            if not df_hev.empty:
                df_hev.to_csv(os.path.join(output_hev_folder, file), index=False)

# Apply to both dynamic data folders
filter_and_save(folder1, gas_folder, hev_folder)
filter_and_save(folder2, gas_folder, hev_folder)

In [ ]:
# Function to get row counts for each CSV file
def get_row_counts(folder_path):
    file_counts = {}
    for file in sorted(os.listdir(folder_path)):  # sorted for consistent order
        if file.endswith('.csv'):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)
            file_counts[file] = len(df)
    return file_counts

# Get row counts for both folders
counts_folder_gas = get_row_counts(gas_folder)
counts_folder_hev = get_row_counts(hev_folder)

# Print Folder 1 details
print("Gasoline folder:")
total_folder_gas = 0
for file, count in counts_folder_gas.items():
    print(f"{file}: {count:,} rows")  # Add commas in number
    total_folder_gas += count
print(f"Total rows in Folder 1: {total_folder_gas:,}\n")

# Print Folder 2 details
print("HEV & PHEV folder:")
total_folder_hev = 0
for file, count in counts_folder_hev.items():
    print(f"{file}: {count:,} rows")
    total_folder_hev += count
print(f"Total rows in Folder 2: {total_folder_hev:,}\n")

# Print combined total
grand_total_gas_hev = total_folder_gas + total_folder_hev
print(f"Grand Total (Folder 1 + Folder 2): {grand_total_gas_hev:,}")

Gasoline folder:
VED_171101_week.csv: 323,615 rows
VED_171108_week.csv: 344,755 rows
VED_171115_week.csv: 369,009 rows
VED_171122_week.csv: 294,642 rows
VED_171129_week.csv: 333,513 rows
VED_171206_week.csv: 388,666 rows
VED_171213_week.csv: 452,921 rows
VED_171220_week.csv: 328,648 rows
VED_171227_week.csv: 184,693 rows
VED_180103_week.csv: 249,135 rows
VED_180110_week.csv: 354,911 rows
VED_180117_week.csv: 349,878 rows
VED_180124_week.csv: 277,280 rows
VED_180131_week.csv: 286,129 rows
VED_180207_week.csv: 256,032 rows
VED_180214_week.csv: 249,175 rows
VED_180221_week.csv: 165,632 rows
VED_180228_week.csv: 261,744 rows
VED_180307_week.csv: 239,692 rows
VED_180314_week.csv: 237,466 rows
VED_180321_week.csv: 237,008 rows
VED_180328_week.csv: 186,951 rows
VED_180404_week.csv: 259,585 rows
VED_180411_week.csv: 261,591 rows
VED_180418_week.csv: 281,556 rows
VED_180425_week.csv: 246,004 rows
VED_180502_week.csv: 247,355 rows
VED_180509_week.csv: 255,353 rows
VED_180516_week.csv: 273,500 ro

In [ ]:
first_file_hev = sorted(os.listdir(hev_folder))[0]  # Get first CSV file name
first_file_hev_path = os.path.join(hev_folder, first_file_hev)
print(f"First 5 rows of {first_file_hev}:\n")
df_first_hev = pd.read_csv(first_file_hev_path)
print(df_first_hev.head())

First 5 rows of VED_171101_week.csv:

     DayNum  VehId  Trip  Timestamp(ms)  Latitude[deg]  Longitude[deg]  \
0  1.058991     11  1485              0      42.300565      -83.735444   
1  1.058991     11  1485            200      42.300565      -83.735444   
2  1.058991     11  1485            300      42.300565      -83.735444   
3  1.058991     11  1485            600      42.300565      -83.735444   
4  1.058991     11  1485           1600      42.300565      -83.735444   

   Vehicle Speed[km/h]  MAF[g/sec]  Engine RPM[RPM]  Absolute Load[%]  ...  \
0            40.439999        6.13           1336.0               NaN  ...   
1            40.439999        4.20           2214.0               NaN  ...   
2            40.439999        4.20           2214.0               NaN  ...   
3            42.599998        4.20           2214.0               NaN  ...   
4            43.459999        4.20           2214.0               NaN  ...   

   Air Conditioning Power[kW]  Air Conditioning 

In [ ]:
print(f"\nColumn names in {first_file_hev}:")
for col in df_first_hev.columns:
    print(col)


Column names in VED_171101_week.csv:
DayNum
VehId
Trip
Timestamp(ms)
Latitude[deg]
Longitude[deg]
Vehicle Speed[km/h]
MAF[g/sec]
Engine RPM[RPM]
Absolute Load[%]
OAT[DegC]
Fuel Rate[L/hr]
Air Conditioning Power[kW]
Air Conditioning Power[Watts]
Heater Power[Watts]
HV Battery Current[A]
HV Battery SOC[%]
HV Battery Voltage[V]
Short Term Fuel Trim Bank 1[%]
Short Term Fuel Trim Bank 2[%]
Long Term Fuel Trim Bank 1[%]
Long Term Fuel Trim Bank 2[%]
